<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🧠 LangGraph Memory —— checkpointer + thread_id

**一句话定位**:graph 默认**无记忆**(每次 `invoke` 都是全新 state);记忆 = 编译时挂一个 **checkpointer**,调用时带同一个 **`thread_id`**。

</div>

<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

**不加会怎样**:第一轮"3+4"得 7,第二轮"乘以 2"——模型完全不知道 `that` 是谁。**state 只活在一次 `invoke` 内**。

</div>

### 1️⃣ 两种记忆:别混

| | 短期记忆 (thread 内) | 长期记忆 (跨 thread) |
|---|---|---|
| 机制 | **Checkpointer** | **Store** |
| 存什么 | 整个 graph state(如 `messages`) | 自由 JSON(用户偏好、事实) |
| 隔离键 | `thread_id` | `namespace`(如 `("memories", user_id)`) |
| 编译参数 | `compile(checkpointer=...)` | `compile(store=...)` |
| 本节重点 | ✅ | 📎 §4 简介 |

### 0️⃣ 前置:本笔记复用的最小 ReAct graph

下面所有代码里的 `builder` / `tools` / `llm` 都来自这里(**还没加记忆**,只是 `StateGraph` 的"设计图",`compile()` 之后才变成可运行的 graph)。
- `builder` = `StateGraph` 构造器,尚未编译
- 记忆的唯一改动点就是**最后 `builder.compile(...)` 时传什么参数**

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

# 1. 工具(普通函数即可)
def add(a: int, b: int) -> int:
    """Adds a and b."""
    return a + b

def multiply(a: int, b: int) -> int:
    """Multiply a and b."""
    return a * b

tools = [add, multiply]

# 2. 模型 + 绑定工具
llm = ChatOpenAI(model="gpt-4o")
llm_with_tools = llm.bind_tools(tools)
sys_msg = SystemMessage("You are a helpful assistant tasked with performing arithmetic.")

# 3. 节点:LLM 决定直接答 or 调工具
def assistant(state: MessagesState):
    return {"messages": [llm_with_tools.invoke([sys_msg] + state["messages"])]}

# 4. 搭图(此时还是 builder,未编译)
builder = StateGraph(MessagesState)
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)   # 有 tool_call → tools,否则 END
builder.add_edge("tools", "assistant")

# 5. 不带记忆编译:每次 invoke 都失忆
react_graph = builder.compile()

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🔑 第二部分:短期记忆(最常用)

</div>

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver   # = MemorySaver(旧名,仍可用)

graph = builder.compile(checkpointer=InMemorySaver())        # ① 同一个 builder,只多传 checkpointer

config = {"configurable": {"thread_id": "1"}}                # ② 会话 ID

graph.invoke({"messages": [("human", "Add 3 and 4.")]}, config)
graph.invoke({"messages": [("human", "Multiply that by 2.")]}, config)   # ③ 同 thread → 记得 7

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**心智模型**:`thread_id` = 对话的**会话 ID**。同 id 多次 `invoke` → 共享并延续同一份 state;换 id → 全新互不干扰的对话。
- 每个**超步 (super-step)** 结束都会写一个 checkpoint
- 后续调用**只需传增量**(新的 `HumanMessage`),历史自动从 checkpoint 读出并经 `add_messages` 合并

</div>

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**易错点**
1. 不传 `config`(缺 `thread_id`)→ 报错:有 checkpointer 就**必须**带 `thread_id`
2. 传了 `thread_id` 却忘了挂 checkpointer → **静默无记忆**
3. `InMemorySaver` 存在进程内存里,**重启即丢**——生产换持久化 saver

</div>

#### 持久化 saver 对照

| Saver | 包 | 场景 |
|---|---|---|
| `InMemorySaver` | `langgraph` 自带 | 学习 / 测试 |
| `SqliteSaver` | `langgraph-checkpoint-sqlite` | 本地单机持久化 |
| `PostgresSaver` | `langgraph-checkpoint-postgres` | 生产 |
| LangGraph Platform / Studio | 自动提供 | 无需自己配 |

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🔍 第三部分:查看 / 回溯 / 修改 checkpoint

</div>

In [ ]:
snap = graph.get_state(config)          # 当前最新 StateSnapshot
snap.values["messages"]                 # 当前 state
snap.next                               # 下一个要跑的节点;() 表示已结束
snap.config["configurable"]["checkpoint_id"]

list(graph.get_state_history(config))   # 全部历史 checkpoint(新→旧)

graph.update_state(config, {"messages": [("human", "手动注入")]})   # 手动改 state(走 reducer)

| API | 用途 |
|---|---|
| `get_state(config)` | 读当前 state、下一节点 |
| `get_state_history(config)` | 时间旅行:拿旧 `checkpoint_id` 传进 config 可**从历史点重跑 / 分叉** |
| `update_state(config, values)` | 手动写 state(同样走 `add_messages` reducer) |

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### ⚡ 第四部分:预构建 agent + 长期记忆简介

</div>

In [ ]:
from langchain.agents import create_agent

agent = create_agent("openai:gpt-4o", tools=[add, multiply],
                     checkpointer=InMemorySaver())          # 一个参数搞定
agent.invoke({"messages": [("human", "Add 3 and 4.")]},
             {"configurable": {"thread_id": "1"}})

In [ ]:
# 长期记忆:Store —— 跨 thread 共享,按 namespace 隔离
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore
from langchain_core.runnables import RunnableConfig

def remember(state: MessagesState, config: RunnableConfig, *, store: BaseStore):   # store 自动注入
    uid = config["configurable"]["user_id"]
    store.put(("memories", uid), "food", {"likes": "pizza"})     # 写:(namespace, key, value)
    items = store.search(("memories", uid))                      # 读:按 namespace 检索
    return {"messages": [("ai", f"已记住 {len(items)} 条")]}

b = StateGraph(MessagesState)
b.add_node("remember", remember)
b.add_edge(START, "remember")
graph2 = b.compile(checkpointer=InMemorySaver(), store=InMemoryStore())

graph2.invoke({"messages": [("human", "我爱吃披萨")]},
              {"configurable": {"thread_id": "1", "user_id": "u1"}})

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**历史会无限增长**:checkpointer 会把 `messages` 全存下来,对话越长 token 越贵。配合 `RemoveMessage` / `trim_messages`(见 02 笔记)裁剪,或做摘要。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### ✨ 一句话带走

</div>

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**短期记忆 = `compile(checkpointer=…)` + 每次 `invoke` 带同一个 `thread_id`;长期记忆 = `compile(store=…)` + namespace。** 没挂 checkpointer = 失忆,`InMemorySaver` 重启即丢,生产换 Sqlite / Postgres。

</div>